In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

import tensorflow as tf
import keras_tuner as kt
import tensorboard
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout,BatchNormalization,Input,GRU
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error,r2_score

In [2]:
df_train = pd.read_csv(r'../data/cleaned/train.csv')
df_test = pd.read_csv(r'../data/cleaned/test.csv')
df_val = pd.read_csv(r'../data/cleaned/val.csv')

In [3]:
with open('target_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [24]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

In [25]:
early_stop = EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True,verbose=1)

In [4]:
def create_sequences(df,seq_length, target_col='PJME_MW',horizon=1):
    """
    Converts a scaled DataFrame into 3D sequence arrays for Keras models.
    """
    # Separate feature columns from Datetime
    feature_cols = [col for col in df.columns if col != 'Datetime']
    target_idx = feature_cols.index(target_col)
    
    data = df[feature_cols].values
    X, y = [], []
    
    for i in range(len(data) - seq_length - horizon + 1):
        X.append(data[i : i + seq_length, :])
        if horizon == 1:
            y.append(data[i + seq_length, target_idx])
        else:
            y.append(data[i + seq_length : i + seq_length + horizon, target_idx])
            
    return np.array(X), np.array(y)

In [5]:
SEQ_LEN = 24
HORIZON = 24

X_train, y_train = create_sequences(df_train, seq_length=SEQ_LEN, horizon=HORIZON)
X_val, y_val     = create_sequences(df_val, seq_length=SEQ_LEN, horizon=HORIZON)
X_test, y_test   = create_sequences(df_test, seq_length=SEQ_LEN, horizon=HORIZON)

print(f"X_train Shape: {X_train.shape}")
print(f"X_test Shape:  {X_test.shape}")

X_train Shape: (101593, 24, 13)
X_test Shape:  (21733, 24, 13)


In [6]:
y_test_mw = scaler.inverse_transform(y_test.reshape(-1, 1))

In [ ]:
#drop out 0.2

In [7]:
model_gru = Sequential([
    GRU(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_gru.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_gru = model_gru.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

E0000 00:00:1786439240.205151  509536 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - loss: 0.0068 - mae: 0.0578 - val_loss: 0.0023 - val_mae: 0.0355
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 28s 17ms/step - loss: 0.0023 - mae: 0.0360 - val_loss: 0.0018 - val_mae: 0.0315
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0019 - mae: 0.0329 - val_loss: 0.0018 - val_mae: 0.0302
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0018 - mae: 0.0317 - val_loss: 0.0017 - val_mae: 0.0297
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 24s 15ms/step - loss: 0.0018 - mae: 0.0310 - val_loss: 0.0016 - val_mae: 0.0286
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0017 - mae: 0.0305 - val_loss: 0.0016 - val_mae: 0.0290
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0017 - mae: 0.0300 - val_loss: 0.0016 - val_mae: 0.0284
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0016 - mae: 0.0296 - val_loss: 0.0015 - val_mae: 0.0281
Epoch 9/10
1588/1588 ━━━

In [8]:
y_pred_gru_scaled = model_gru.predict(X_test)

y_pred_gru_mw = scaler.inverse_transform(y_pred_gru_scaled.reshape(-1, 1))

mae_gru = mean_absolute_error(y_test_mw, y_pred_gru_mw)
rmse_gru = np.sqrt(mean_squared_error(y_test_mw, y_pred_gru_mw))
mape_gru = np.mean(np.abs((y_test_mw - y_pred_gru_mw) / y_test_mw)) * 100
r2_gru = r2_score(y_test_mw, y_pred_gru_mw)

print(f"MAE:  {mae_gru:.2f} MW")
print(f"RMSE: {rmse_gru:.2f} MW")
print(f"MAPE: {mape_gru:.2f}%")
print(f"R2:   {r2_gru:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  1438.84 MW
RMSE: 2002.50 MW
MAPE: 4.51%
R2:   0.9034


In [9]:
model_gru.save(r'../models/early_rnn.keras')
history_df =pd.DataFrame(history_gru.history)

history_df.to_csv(r'../log/early_rnn.csv',index=False)

In [ ]:
#0.3

In [10]:
model_gru = Sequential([
    GRU(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_gru.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_gru = model_gru.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.0091 - mae: 0.0659 - val_loss: 0.0027 - val_mae: 0.0399
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0027 - mae: 0.0396 - val_loss: 0.0019 - val_mae: 0.0323
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0022 - mae: 0.0352 - val_loss: 0.0018 - val_mae: 0.0307
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0020 - mae: 0.0339 - val_loss: 0.0018 - val_mae: 0.0303
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0019 - mae: 0.0330 - val_loss: 0.0017 - val_mae: 0.0307
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0019 - mae: 0.0323 - val_loss: 0.0017 - val_mae: 0.0298
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0018 - mae: 0.0318 - val_loss: 0.0016 - val_mae: 0.0295
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0018 - mae: 0.0314 - val_loss: 0.0016 - val_mae: 0.0288
Epoch 9/10
1588/1588 ━━━

In [11]:
y_pred_gru_scaled = model_gru.predict(X_test)

y_pred_gru_mw = scaler.inverse_transform(y_pred_gru_scaled.reshape(-1, 1))

mae_gru = mean_absolute_error(y_test_mw, y_pred_gru_mw)
rmse_gru = np.sqrt(mean_squared_error(y_test_mw, y_pred_gru_mw))
mape_gru = np.mean(np.abs((y_test_mw - y_pred_gru_mw) / y_test_mw)) * 100
r2_gru = r2_score(y_test_mw, y_pred_gru_mw)

print(f"MAE:  {mae_gru:.2f} MW")
print(f"RMSE: {rmse_gru:.2f} MW")
print(f"MAPE: {mape_gru:.2f}%")
print(f"R2:   {r2_gru:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  1489.41 MW
RMSE: 2028.77 MW
MAPE: 4.77%
R2:   0.9008


In [ ]:
#0.5

In [12]:
model_gru = Sequential([
    GRU(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_gru.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_gru = model_gru.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - loss: 0.0096 - mae: 0.0683 - val_loss: 0.0028 - val_mae: 0.0401
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.0032 - mae: 0.0433 - val_loss: 0.0022 - val_mae: 0.0353
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.0027 - mae: 0.0395 - val_loss: 0.0019 - val_mae: 0.0326
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0025 - mae: 0.0382 - val_loss: 0.0019 - val_mae: 0.0316
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0024 - mae: 0.0373 - val_loss: 0.0018 - val_mae: 0.0309
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0024 - mae: 0.0368 - val_loss: 0.0017 - val_mae: 0.0309
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0023 - mae: 0.0362 - val_loss: 0.0018 - val_mae: 0.0308
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0023 - mae: 0.0358 - val_loss: 0.0018 - val_mae: 0.0310
Epoch 9/10
1588/1588 ━━━

In [13]:
y_pred_gru_scaled = model_gru.predict(X_test)

y_pred_gru_mw = scaler.inverse_transform(y_pred_gru_scaled.reshape(-1, 1))

mae_gru = mean_absolute_error(y_test_mw, y_pred_gru_mw)
rmse_gru = np.sqrt(mean_squared_error(y_test_mw, y_pred_gru_mw))
mape_gru = np.mean(np.abs((y_test_mw - y_pred_gru_mw) / y_test_mw)) * 100
r2_gru = r2_score(y_test_mw, y_pred_gru_mw)

print(f"MAE:  {mae_gru:.2f} MW")
print(f"RMSE: {rmse_gru:.2f} MW")
print(f"MAPE: {mape_gru:.2f}%")
print(f"R2:   {r2_gru:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  1598.60 MW
RMSE: 2158.66 MW
MAPE: 5.06%
R2:   0.8877


In [ ]:
#batch normalization

In [15]:
model_gru = Sequential([
    GRU(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    BatchNormalization(),
    Dropout(0.5),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dense(HORIZON)
])

model_gru.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_gru = model_gru.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10


/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - loss: 0.0563 - mae: 0.1377 - val_loss: 0.0045 - val_mae: 0.0535
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0055 - mae: 0.0579 - val_loss: 0.0072 - val_mae: 0.0719
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0041 - mae: 0.0496 - val_loss: 0.0027 - val_mae: 0.0395
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 15ms/step - loss: 0.0036 - mae: 0.0466 - val_loss: 0.0029 - val_mae: 0.0403
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.0033 - mae: 0.0442 - val_loss: 0.0028 - val_mae: 0.0397
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.0031 - mae: 0.0428 - val_loss: 0.0022 - val_mae: 0.0348
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.0029 - mae: 0.0414 - val_loss: 0.0021 - val_mae: 0.0350
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0028 - mae: 0.0406 - val_loss: 0.0021 - val_mae: 0.0359
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━━

In [16]:
y_pred_gru_scaled = model_gru.predict(X_test)

y_pred_gru_mw = scaler.inverse_transform(y_pred_gru_scaled.reshape(-1, 1))

mae_gru = mean_absolute_error(y_test_mw, y_pred_gru_mw)
rmse_gru = np.sqrt(mean_squared_error(y_test_mw, y_pred_gru_mw))
mape_gru = np.mean(np.abs((y_test_mw - y_pred_gru_mw) / y_test_mw)) * 100
r2_gru = r2_score(y_test_mw, y_pred_gru_mw)

print(f"MAE:  {mae_gru:.2f} MW")
print(f"RMSE: {rmse_gru:.2f} MW")
print(f"MAPE: {mape_gru:.2f}%")
print(f"R2:   {r2_gru:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  1652.86 MW
RMSE: 2270.09 MW
MAPE: 5.10%
R2:   0.8759


In [ ]:
#optimizers

In [ ]:
#adam

In [17]:
model_gru = Sequential([
    GRU(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_gru.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_gru = model_gru.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10


/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 0.0052 - mae: 0.0490 - val_loss: 0.0022 - val_mae: 0.0340
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0017 - mae: 0.0306 - val_loss: 0.0020 - val_mae: 0.0327
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0016 - mae: 0.0287 - val_loss: 0.0018 - val_mae: 0.0301
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0015 - mae: 0.0277 - val_loss: 0.0017 - val_mae: 0.0292
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.0014 - mae: 0.0270 - val_loss: 0.0016 - val_mae: 0.0290
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0014 - mae: 0.0266 - val_loss: 0.0016 - val_mae: 0.0285
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 15ms/step - loss: 0.0013 - mae: 0.0261 - val_loss: 0.0016 - val_mae: 0.0293
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0013 - mae: 0.0258 - val_loss: 0.0015 - val_mae: 0.0274
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━━

In [18]:
y_pred_gru_scaled = model_gru.predict(X_test)

y_pred_gru_mw = scaler.inverse_transform(y_pred_gru_scaled.reshape(-1, 1))

mae_gru = mean_absolute_error(y_test_mw, y_pred_gru_mw)
rmse_gru = np.sqrt(mean_squared_error(y_test_mw, y_pred_gru_mw))
mape_gru = np.mean(np.abs((y_test_mw - y_pred_gru_mw) / y_test_mw)) * 100
r2_gru = r2_score(y_test_mw, y_pred_gru_mw)

print(f"MAE:  {mae_gru:.2f} MW")
print(f"RMSE: {rmse_gru:.2f} MW")
print(f"MAPE: {mape_gru:.2f}%")
print(f"R2:   {r2_gru:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  1388.34 MW
RMSE: 1954.88 MW
MAPE: 4.40%
R2:   0.9079


In [19]:
model_gru.save(r'../models/gru_adam.keras')
history_df =pd.DataFrame(history_gru.history)

history_df.to_csv(r'../log/gru_adam.csv',index=False)

In [ ]:
#sgd

In [20]:
model_gru = Sequential([
    GRU(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_gru.compile(optimizer='sgd', loss='mse', metrics=['mae'])


history_gru = model_gru.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10


/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 0.0403 - mae: 0.1500 - val_loss: 0.0200 - val_mae: 0.1149
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0164 - mae: 0.1005 - val_loss: 0.0165 - val_mae: 0.1035
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0145 - mae: 0.0946 - val_loss: 0.0142 - val_mae: 0.0954
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0134 - mae: 0.0906 - val_loss: 0.0127 - val_mae: 0.0897
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0125 - mae: 0.0876 - val_loss: 0.0117 - val_mae: 0.0856
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0118 - mae: 0.0853 - val_loss: 0.0109 - val_mae: 0.0823
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0113 - mae: 0.0834 - val_loss: 0.0103 - val_mae: 0.0800
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.0109 - mae: 0.0817 - val_loss: 0.0099 - val_mae: 0.0782
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━━━

In [21]:
y_pred_gru_scaled = model_gru.predict(X_test)

y_pred_gru_mw = scaler.inverse_transform(y_pred_gru_scaled.reshape(-1, 1))

mae_gru = mean_absolute_error(y_test_mw, y_pred_gru_mw)
rmse_gru = np.sqrt(mean_squared_error(y_test_mw, y_pred_gru_mw))
mape_gru = np.mean(np.abs((y_test_mw - y_pred_gru_mw) / y_test_mw)) * 100
r2_gru = r2_score(y_test_mw, y_pred_gru_mw)

print(f"MAE:  {mae_gru:.2f} MW")
print(f"RMSE: {rmse_gru:.2f} MW")
print(f"MAPE: {mape_gru:.2f}%")
print(f"R2:   {r2_gru:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  3627.66 MW
RMSE: 4637.41 MW
MAPE: 12.06%
R2:   0.4819


In [ ]:
#rms prop

In [22]:
model_gru = Sequential([
    GRU(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_gru.compile(optimizer='RMSprop', loss='mse', metrics=['mae'])


history_gru = model_gru.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10


/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.0058 - mae: 0.0575 - val_loss: 0.0034 - val_mae: 0.0452
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.0028 - mae: 0.0407 - val_loss: 0.0028 - val_mae: 0.0411
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0022 - mae: 0.0356 - val_loss: 0.0021 - val_mae: 0.0342
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 12ms/step - loss: 0.0020 - mae: 0.0334 - val_loss: 0.0021 - val_mae: 0.0346
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0019 - mae: 0.0322 - val_loss: 0.0021 - val_mae: 0.0344
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 24s 15ms/step - loss: 0.0018 - mae: 0.0313 - val_loss: 0.0021 - val_mae: 0.0347
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0017 - mae: 0.0307 - val_loss: 0.0020 - val_mae: 0.0328
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0017 - mae: 0.0302 - val_loss: 0.0018 - val_mae: 0.0309
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━━

In [23]:
y_pred_gru_scaled = model_gru.predict(X_test)

y_pred_gru_mw = scaler.inverse_transform(y_pred_gru_scaled.reshape(-1, 1))

mae_gru = mean_absolute_error(y_test_mw, y_pred_gru_mw)
rmse_gru = np.sqrt(mean_squared_error(y_test_mw, y_pred_gru_mw))
mape_gru = np.mean(np.abs((y_test_mw - y_pred_gru_mw) / y_test_mw)) * 100
r2_gru = r2_score(y_test_mw, y_pred_gru_mw)

print(f"MAE:  {mae_gru:.2f} MW")
print(f"RMSE: {rmse_gru:.2f} MW")
print(f"MAPE: {mape_gru:.2f}%")
print(f"R2:   {r2_gru:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  1527.82 MW
RMSE: 2132.19 MW
MAPE: 4.80%
R2:   0.8905


In [ ]:
#early stopping

In [26]:
model_gru = Sequential([
    GRU(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_gru.compile(optimizer='RMSprop', loss='mse', metrics=['mae'])


history_gru = model_gru.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
    callbacks=[early_stop]
)

Epoch 1/10


/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 0.0067 - mae: 0.0601 - val_loss: 0.0039 - val_mae: 0.0477
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.0029 - mae: 0.0416 - val_loss: 0.0028 - val_mae: 0.0397
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.0025 - mae: 0.0377 - val_loss: 0.0024 - val_mae: 0.0371
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.0023 - mae: 0.0357 - val_loss: 0.0022 - val_mae: 0.0350
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.0020 - mae: 0.0338 - val_loss: 0.0022 - val_mae: 0.0350
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 24s 15ms/step - loss: 0.0019 - mae: 0.0320 - val_loss: 0.0021 - val_mae: 0.0340
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0018 - mae: 0.0311 - val_loss: 0.0021 - val_mae: 0.0327
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0017 - mae: 0.0304 - val_loss: 0.0023 - val_mae: 0.0360
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━━

In [27]:
y_pred_gru_scaled = model_gru.predict(X_test)

y_pred_gru_mw = scaler.inverse_transform(y_pred_gru_scaled.reshape(-1, 1))

mae_gru = mean_absolute_error(y_test_mw, y_pred_gru_mw)
rmse_gru = np.sqrt(mean_squared_error(y_test_mw, y_pred_gru_mw))
mape_gru = np.mean(np.abs((y_test_mw - y_pred_gru_mw) / y_test_mw)) * 100
r2_gru = r2_score(y_test_mw, y_pred_gru_mw)

print(f"MAE:  {mae_gru:.2f} MW")
print(f"RMSE: {rmse_gru:.2f} MW")
print(f"MAPE: {mape_gru:.2f}%")
print(f"R2:   {r2_gru:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
MAE:  1571.57 MW
RMSE: 2168.76 MW
MAPE: 5.04%
R2:   0.8867


In [ ]:
#learning rate

In [28]:
model_gru = Sequential([
    GRU(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_gru.compile(optimizer='RMSprop', loss='mse', metrics=['mae'])


history_gru = model_gru.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
    callbacks=[lr_scheduler]
)

/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 22s 12ms/step - loss: 0.0073 - mae: 0.0627 - val_loss: 0.0035 - val_mae: 0.0462 - learning_rate: 0.0010
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0031 - mae: 0.0427 - val_loss: 0.0041 - val_mae: 0.0493 - learning_rate: 0.0010
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0025 - mae: 0.0379 - val_loss: 0.0028 - val_mae: 0.0413 - learning_rate: 0.0010
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0022 - mae: 0.0355 - val_loss: 0.0022 - val_mae: 0.0346 - learning_rate: 0.0010
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0020 - mae: 0.0337 - val_loss: 0.0021 - val_mae: 0.0339 - learning_rate: 0.0010
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 25s 15ms/step - loss: 0.0019 - mae: 0.0320 - val_loss: 0.0019 - val_mae: 0.0316 - learning_rate: 0.0010
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 15ms/step - loss: 0.0018 - mae: 0.0309 - val_loss: 0.0021 - val_mae: 0.0341 - 

In [29]:
y_pred_gru_scaled = model_gru.predict(X_test)

y_pred_gru_mw = scaler.inverse_transform(y_pred_gru_scaled.reshape(-1, 1))

mae_gru = mean_absolute_error(y_test_mw, y_pred_gru_mw)
rmse_gru = np.sqrt(mean_squared_error(y_test_mw, y_pred_gru_mw))
mape_gru = np.mean(np.abs((y_test_mw - y_pred_gru_mw) / y_test_mw)) * 100
r2_gru = r2_score(y_test_mw, y_pred_gru_mw)

print(f"MAE:  {mae_gru:.2f} MW")
print(f"RMSE: {rmse_gru:.2f} MW")
print(f"MAPE: {mape_gru:.2f}%")
print(f"R2:   {r2_gru:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  1773.74 MW
RMSE: 2348.85 MW
MAPE: 5.84%
R2:   0.8671


In [ ]:
#batch size 32

In [31]:
model_gru = Sequential([
    GRU(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_gru.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_gru = model_gru.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[lr_scheduler]
)

Epoch 1/10
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 35s 11ms/step - loss: 0.0043 - mae: 0.0444 - val_loss: 0.0020 - val_mae: 0.0332 - learning_rate: 0.0010
Epoch 2/10
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 34s 11ms/step - loss: 0.0017 - mae: 0.0299 - val_loss: 0.0018 - val_mae: 0.0303 - learning_rate: 0.0010
Epoch 3/10
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 34s 11ms/step - loss: 0.0015 - mae: 0.0281 - val_loss: 0.0018 - val_mae: 0.0313 - learning_rate: 0.0010
Epoch 4/10
3172/3175 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0014 - mae: 0.0273
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 33s 10ms/step - loss: 0.0014 - mae: 0.0273 - val_loss: 0.0017 - val_mae: 0.0297 - learning_rate: 0.0010
Epoch 5/10
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 34s 11ms/step - loss: 0.0013 - mae: 0.0257 - val_loss: 0.0015 - val_mae: 0.0277 - learning_rate: 5.0000e-04
Epoch 6/10
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 34s 11ms/step - loss: 0.0013 - mae: 0.0253 - val_loss: 0.0015 - val_mae: 0.

In [32]:
y_pred_gru_scaled = model_gru.predict(X_test)

y_pred_gru_mw = scaler.inverse_transform(y_pred_gru_scaled.reshape(-1, 1))

mae_gru = mean_absolute_error(y_test_mw, y_pred_gru_mw)
rmse_gru = np.sqrt(mean_squared_error(y_test_mw, y_pred_gru_mw))
mape_gru = np.mean(np.abs((y_test_mw - y_pred_gru_mw) / y_test_mw)) * 100
r2_gru = r2_score(y_test_mw, y_pred_gru_mw)

print(f"MAE:  {mae_gru:.2f} MW")
print(f"RMSE: {rmse_gru:.2f} MW")
print(f"MAPE: {mape_gru:.2f}%")
print(f"R2:   {r2_gru:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  1386.11 MW
RMSE: 1943.91 MW
MAPE: 4.42%
R2:   0.9090


In [33]:
model_gru.save(r'../models/gru_32.keras')
history_df =pd.DataFrame(history_gru.history)

history_df.to_csv(r'../log/gru_32.csv',index=False)

In [ ]:
#16

In [34]:
model_gru = Sequential([
    GRU(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_gru.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_gru = model_gru.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=16,
)

/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 65s 10ms/step - loss: 0.0030 - mae: 0.0382 - val_loss: 0.0019 - val_mae: 0.0317
Epoch 2/10
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 66s 10ms/step - loss: 0.0016 - mae: 0.0288 - val_loss: 0.0021 - val_mae: 0.0354
Epoch 3/10
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 73s 12ms/step - loss: 0.0014 - mae: 0.0274 - val_loss: 0.0016 - val_mae: 0.0286
Epoch 4/10
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 56s 9ms/step - loss: 0.0014 - mae: 0.0265 - val_loss: 0.0015 - val_mae: 0.0271
Epoch 5/10
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 60s 9ms/step - loss: 0.0013 - mae: 0.0258 - val_loss: 0.0015 - val_mae: 0.0271
Epoch 6/10
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 57s 9ms/step - loss: 0.0012 - mae: 0.0251 - val_loss: 0.0016 - val_mae: 0.0299
Epoch 7/10
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 53s 8ms/step - loss: 0.0012 - mae: 0.0246 - val_loss: 0.0014 - val_mae: 0.0263
Epoch 8/10
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 55s 9ms/step - loss: 0.0012 - mae: 0.0243 - val_loss: 0.0015 - val_mae: 0.0277
Epoch 9/10
6350/6350 ━━━━━━━━

In [35]:
y_pred_gru_scaled = model_gru.predict(X_test)

y_pred_gru_mw = scaler.inverse_transform(y_pred_gru_scaled.reshape(-1, 1))

mae_gru = mean_absolute_error(y_test_mw, y_pred_gru_mw)
rmse_gru = np.sqrt(mean_squared_error(y_test_mw, y_pred_gru_mw))
mape_gru = np.mean(np.abs((y_test_mw - y_pred_gru_mw) / y_test_mw)) * 100
r2_gru = r2_score(y_test_mw, y_pred_gru_mw)

print(f"MAE:  {mae_gru:.2f} MW")
print(f"RMSE: {rmse_gru:.2f} MW")
print(f"MAPE: {mape_gru:.2f}%")
print(f"R2:   {r2_gru:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  1338.70 MW
RMSE: 1915.71 MW
MAPE: 4.18%
R2:   0.9116


In [36]:
model_gru.save(r'../models/gru_16.keras')
history_df =pd.DataFrame(history_gru.history)

history_df.to_csv(r'../log/gru_16.csv',index=False)

In [ ]:
#number of layers

In [37]:
model_gru = Sequential([
    GRU(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),

    GRU(32, return_sequences=False),
    Dropout(0.2),
    
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_gru.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_gru = model_gru.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
)

/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 35s 20ms/step - loss: 0.0073 - mae: 0.0596 - val_loss: 0.0025 - val_mae: 0.0376
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 32s 20ms/step - loss: 0.0026 - mae: 0.0386 - val_loss: 0.0019 - val_mae: 0.0321
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 33s 21ms/step - loss: 0.0022 - mae: 0.0355 - val_loss: 0.0019 - val_mae: 0.0317
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 35s 22ms/step - loss: 0.0021 - mae: 0.0344 - val_loss: 0.0017 - val_mae: 0.0308
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 33s 21ms/step - loss: 0.0020 - mae: 0.0335 - val_loss: 0.0017 - val_mae: 0.0298
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 33s 21ms/step - loss: 0.0019 - mae: 0.0326 - val_loss: 0.0016 - val_mae: 0.0292
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 32s 20ms/step - loss: 0.0018 - mae: 0.0317 - val_loss: 0.0018 - val_mae: 0.0307
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 31s 19ms/step - loss: 0.0017 - mae: 0.0309 - val_loss: 0.0018 - val_mae: 0.0310
Epoch 9/10
1588/1588 ━━━

In [38]:
y_pred_gru_scaled = model_gru.predict(X_test)

y_pred_gru_mw = scaler.inverse_transform(y_pred_gru_scaled.reshape(-1, 1))

mae_gru = mean_absolute_error(y_test_mw, y_pred_gru_mw)
rmse_gru = np.sqrt(mean_squared_error(y_test_mw, y_pred_gru_mw))
mape_gru = np.mean(np.abs((y_test_mw - y_pred_gru_mw) / y_test_mw)) * 100
r2_gru = r2_score(y_test_mw, y_pred_gru_mw)

print(f"MAE:  {mae_gru:.2f} MW")
print(f"RMSE: {rmse_gru:.2f} MW")
print(f"MAPE: {mape_gru:.2f}%")
print(f"R2:   {r2_gru:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
MAE:  1622.88 MW
RMSE: 2196.55 MW
MAPE: 5.05%
R2:   0.8838


In [ ]:
#hyper parameter

In [39]:
input_shape = (X_train.shape[1], X_train.shape[2])

def build_model(hp):

    model = Sequential([
        Input(shape=input_shape),

        GRU(
            units=hp.Choice(
                "gru_units",
                [32, 64, 128]
            )
        ),

        BatchNormalization(),

        Dense(
            units=hp.Choice(
                "dense_units",
                [16, 32, 64]
            ),
            activation="relu"
        ),

        Dropout(
            hp.Choice(
                "dropout",
                [0.2, 0.3, 0.5]
            )
        ),

        Dense(HORIZON)
    ])

    model.compile(
         optimizer=hp.Choice(
        "optimizer",
        values=["adam", "rmsprop", "sgd"]),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [40]:
tuner = kt.RandomSearch(
    build_model,
    objective="val_loss",
    max_trials=5,
    directory="enn_tuning",
    project_name="gru_forecasting"
)

In [41]:
tuner.search(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Trial 5 Complete [00h 02m 27s]
val_loss: 0.008330438286066055

Best val_loss So Far: 0.0018292816821485758
Total elapsed time: 00h 22m 09s


In [42]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("GRU units:", best_hp.get("gru_units"))
print("Dense units:", best_hp.get("dense_units"))
print("Dropout:", best_hp.get("dropout"))

GRU units: 128
Dense units: 32
Dropout: 0.2


In [43]:
best_model = tuner.get_best_models(num_models=1)[0]

/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 20 variables. 
  saveable.load_own_variables(store)


In [44]:
y_pred_bilstm_scaled = best_model.predict(X_test)

y_pred_rnn_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_rnn  = mean_absolute_error(y_test_mw, y_pred_rnn_mw)
rmse_rnn = np.sqrt(mean_squared_error(y_test_mw, y_pred_rnn_mw))
mape_rnn = np.mean(np.abs((y_test_mw - y_pred_rnn_mw) / y_test_mw)) * 100
r2_rnn  = r2_score(y_test_mw, y_pred_rnn_mw)

print(f"MAE:  {mae_rnn:.2f} MW")
print(f"RMSE: {rmse_rnn:.2f} MW")
print(f"MAPE: {mape_rnn:.2f}%")
print(f"R2:   {r2_rnn:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
MAE:  1591.72 MW
RMSE: 2145.07 MW
MAPE: 5.03%
R2:   0.8891
